In [ ]:
!pip install gensim

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
def preprocess_text(text):
    import re, string
    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer

    word_tokens = word_tokenize(text)
    word_tokens = [word.lower() for word in word_tokens]
    word_tokens = [re.sub(r'[^\w\s]', '', token) for token in word_tokens if re.sub(r'[^\w\s]', '', token)]
    corpus_stop_words = set(stopwords.words('english'))
    word_tokens = [word for word in word_tokens if not word in corpus_stop_words]
    lemmatizer = WordNetLemmatizer()
    word_tokens = [lemmatizer.lemmatize(word) for word in word_tokens]
    return word_tokens

In [ ]:
text = 'Hello everyone. How are you? Today is a sunny day and better than yesterday.'
word_tokens = preprocess_text(text)
print(word_tokens)

### Bag of Words

In [ ]:
docs = []
docs.append('Hello everyone. How are you? Today is a sunny day and better than yesterday.')
docs.append("How's everyone doing, today?  It is not sunny today.")
docs

##### Preprocess and Counter

In [ ]:
# Preprocess each documents
processed_docs = [preprocess_text(doc) for doc in docs]
print(processed_docs)

# Flatten list
all_tokens = [word for doc in processed_docs for word in doc]
print(all_tokens)

# Unique sorted vocabulary
vocabs = sorted(set(all_tokens))
print(vocabs)

In [ ]:
def create_bow_vector(tokens, vocabs):
    from collections import Counter
    word_count = Counter(tokens)
    return [word_count[word] for word in vocabs]

# Count frequencies of unique words in each document
bow_freqs = [create_bow_vector(doc, vocabs) for doc in processed_docs]
bow_vectors = pd.DataFrame(bow_freqs, columns=vocabs)
print(bow_vectors)

##### CountVectorizer

In [ ]:
# Create Bag of Words using CountVectorizer
# CountVectorizer tokenizes, lowercases (by default), and creates document-term matrix
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
result = vectorizer.fit_transform(docs)

# Show Vocabulary
print("Vocabulary:\n", vectorizer.vocabulary_)  # value in dictionary is word index in matrix result
# Show vocab frequency in each document
print(result.toarray())

In [ ]:
# Rearrange outputs to see word frequencies in each document
vocabs = {value: key for key, value in vectorizer.vocabulary_.items()}
feature_names = []
for i in range(len(vocabs)):
    feature_names.append(vocabs[i])

bow_vectors2 = pd.DataFrame(result.toarray(), columns=feature_names)
print(bow_vectors2)

### N-Grams

In [ ]:
docs = []
docs.append('Machine learning models learn from data.')
docs.append('Deep learning models require large data.')
docs.append('Machine learning improves decision making.')
docs.append('Data science uses machine learning techniques.')
docs.append('Large data improves machine performance.')
docs

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(ngram_range=(1,2))  # unigram + bigram
result = vectorizer.fit_transform(docs)

# Show Vocabulary
print("Vocabulary:\n", vectorizer.vocabulary_)  # value in dictionary is word index in matrix result
# Show vocab frequency in each document
print(result.toarray())

In [ ]:
# Rearrange outputs to see word frequencies in each document
vocabs = {value: key for key, value in vectorizer.vocabulary_.items()}
feature_names = []
for i in range(len(vocabs)):
    feature_names.append(vocabs[i])

ngrams_vector = pd.DataFrame(result.toarray(), columns=feature_names)
print(ngrams_vector)

### TF-IDF

In [ ]:
docs = []
docs.append('It is going to rain today.')
docs.append('Today I am not going outside.')
docs.append('I am going to watch TV.')
docs

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')
result = tfidf.fit_transform(docs)
print(tfidf.get_feature_names_out())
print(result.toarray())

In [ ]:
feature_names = tfidf.get_feature_names_out()
tfidf_vectors = pd.DataFrame(result.toarray(), columns=feature_names)
print(tfidf_vectors)

### Word2Vec

In [ ]:
docs = []
docs.append('A man lives with two cats and a dog in his apartment.')
docs.append('Inside his apartment, there are a bed, a table and four chairs.')
docs.append('The man is sleeping on the bed.')
docs.append('One cat stands on the table.')
docs.append('The other cat sits on one chair.')
docs.append('The dog sleeps on the floor.')
docs

In [ ]:
# Preprocess documents
processed_docs = [preprocess_text(doc) for doc in docs]
print(processed_docs)

In [ ]:
# Flatten list
all_tokens = [word for doc in processed_docs for word in doc]
print(all_tokens)

# Unique sorted vocabulary
vocabs = sorted(set(all_tokens))
print(vocabs)
print(len(vocabs))

#### Word2Vec CBOW Model

In [ ]:
# Note that Word2Vec takes nested list of words as first paarameter
# vector_size (int, optional) – Dimensionality of the word vectors.
# window (int, optional) – Maximum distance between the current and predicted word within a sentence.
# min_count (int, optional) – Ignores all words with total frequency lower than this.
# sg ({0, 1}, optional) – 1 = skip gram, 0 = CBOW

from gensim.models import Word2Vec
model = Word2Vec(sentences=processed_docs,
                 vector_size=100,   # size of word embeddings
                 window=5,          # context window size
                 min_count=1,       # include all words
                 workers=4,         # number of CPU threads used to train the model in parallel.
                 sg=0               # CBOW
                )

In [ ]:
# Get all possible words from Word2Vec model
vocab_list = list(model.wv.key_to_index.keys())
print(len(vocab_list))
vocab_list

In [ ]:
vector = model.wv['cat']
print("Vector length:", len(vector))   # output will be a 100-dimensional vector.
print(vector)

In [ ]:
# Find most similar words to the query word
query_word = 'cat'
similar_words = model.wv.most_similar(query_word)
print(similar_words)

In [ ]:
# Find Similarity between 2 words
word1 = 'cat'
word2 = 'dog'
similarity_score = model.wv.similarity(word1, word2)
print(similarity_score)

In [ ]:
# Find Similarity between 2 words
word1 = 'cat'
word2 = 'man'
similarity_score = model.wv.similarity(word1, word2)
print(similarity_score)

In [ ]:
def compute_cosine_similarity(vector1, vector2):
    squared_first_sum = sum([val*val for val in vector1])**0.5
    squared_second_sum = sum([val*val for val in vector2])**0.5
    cosine_similar = np.dot(vector1, vector2) / (squared_first_sum*squared_second_sum)
    return cosine_similar

# Find Similarity between 2 word vectors using cosine similarity
vector1 = model.wv['cat']
vector2 = model.wv['dog']
cosine_similar = compute_cosine_similarity(vector1, vector2)
print(cosine_similar)

In [ ]:
# Find Similarity between 2 word vectors using cosine similarity
vector1 = model.wv['cat']
vector2 = model.wv['man']
cosine_similar = compute_cosine_similarity(vector1, vector2)
print(cosine_similar)

In [ ]:
# See a list of top 10 most positively similar word to cat (default size of list = 10)
model.wv.most_similar(positive=['cat'])

In [ ]:
# See a list of top 5 most positively similar word to dog
model.wv.most_similar(positive=['dog'], topn=5)

In [ ]:
# See a list of top 10 most negatively similar word to cat (default size of list = 10)
model.wv.most_similar(negative=['cat'])

In [ ]:
# See a list of top 5 most negatively similar word to dog
model.wv.most_similar(negative=['dog'], topn=5)

In [ ]:
# Select word that least match the remaining words in the list
print(model.wv.doesnt_match(['dog', 'man', 'cat']))
similarity_score1 = model.wv.similarity('dog', 'man')
similarity_score2 = model.wv.similarity('man', 'cat')
similarity_score3 = model.wv.similarity('dog', 'cat')
print(similarity_score1, similarity_score2, similarity_score3)

In [ ]:
# Plot similarities between each word

# Source: https://medium.com/@manansuri/a-dummys-guide-to-word2vec-456444f3c673
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
%matplotlib inline
def display_pca_scatterplot(model, words=None, sample=0):
    if words == None:
        if sample > 0:
            words = np.random.choice(list(model.vocab.keys()), sample)
        else:
            words = [ word for word in model.vocab ]

    word_vectors = np.array([model[w] for w in words])

    twodim = PCA().fit_transform(word_vectors)[:,:2]

    plt.figure(figsize=(10,10))
    plt.scatter(twodim[:,0], twodim[:,1], edgecolors='k', c='r')
    for word, (x,y) in zip(words, twodim):
        #plt.text(x+0.05, y+0.05, word)
        plt.text(x, y, word)
    plt.show()

display_pca_scatterplot(model.wv, vocab_list)

In [ ]:
# Predict the target word, from the context word
# Based on P(target word∣context words)
# Work with CBOW model only

# Probability is computed by
# 1. Takes and averages context word vectors
# 2. Multiplies by output weight matrix W
# 3. Applies softmax
# 4. Returns highest probability words
model.predict_output_word(['cat'])

In [ ]:
# Predict the target word, from the list of context words
model.predict_output_word(['cat', 'dog'])

##### Model can be saved and loaded to train later

In [ ]:
# Model can be saved and reload to use later
model.save('cbow_model')

In [ ]:
# Load model and continue training
new_model = Word2Vec.load('cbow_model')
new_texts = ['The man brings three chickens into the apartment']
new_processed_docs = [preprocess_text(doc) for doc in new_texts]
# Flatten list
new_processed_tokens = [word for doc in new_processed_docs for word in doc]
print(new_processed_tokens)

In [ ]:
# Build new vocab
# Compare last and new vocab lists
new_model.build_vocab([new_processed_tokens], update=True)
new_vocab_list = list(new_model.wv.key_to_index.keys())
print(len(vocab_list))
print(len(new_vocab_list))
new_vocab_list

In [ ]:
new_model.train(new_processed_tokens, total_examples=new_model.corpus_count, epochs=model.epochs)

In [ ]:
# Find Similarity between 2 words
new_model.wv.similarity('cat', 'chicken')

In [ ]:
new_model.predict_output_word(['chicken'])

#### Word2Vec Skip-Gram Model

In [ ]:
sg_model = Word2Vec(sentences=processed_docs,
                    vector_size=100,
                    window=5,
                    min_count=1,
                    sg=1   # Skip-gram
                   )

In [ ]:
# Get all possible words from Word2Vec model
vocab_list = list(sg_model.wv.key_to_index.keys())
vocab_list

In [ ]:
# Find most similar words to the query word
query_word = 'cat'
similar_words = sg_model.wv.most_similar(query_word)
print(similar_words)

In [ ]:
# Find Similarity between 2 words
word1 = 'cat'
word2 = 'dog'
similarity_score = sg_model.wv.similarity(word1, word2)
print(similarity_score)

In [ ]:
# Find Similarity between 2 words
word1 = 'cat'
word2 = 'table'
similarity_score = model.wv.similarity(word1, word2)
print(similarity_score)